To establish a bidirectional connection with an interface widget, you must pass the tracking instance using explicit initialization attributes:


•	textvariable=: For data streams. Tells widgets like an Entry input field or a Label display to link their character strings directly to a tk.StringVar()

.
•	variable=: For tracking binary states. Tells operational toggle widgets like a Checkbutton to track whether the element is selected (True) or clear (False) using a tk.BooleanVar().
 provided.



In [1]:
import tkinter as tk

class VariableBindingApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Diagnostic: Variable Binding")
        self.geometry("450x300")

        # REACTIVE DATA: The central "Source of Truth"
        self.text_val = tk.StringVar(value="Type here...")

        tk.Label(self, text="1. Enter text to test live sync:").pack()
        self.ent = tk.Entry(self, textvariable=self.text_val)
        self.ent.pack(pady=5)

        # WRONG WAY: Label only receives initial value. It does not track.
        # We explicitly label this as the "Static/Broken" binding.
        tk.Label(self, text="WRONG WAY: Using 'text=' parameter", 
                 fg="red", font=("Arial", 8, "bold")).pack(pady=(10, 0))
        lbl_w = tk.Label(self, text=self.text_val, fg="red")
        lbl_w.pack(pady=2)
        
        # CORRECT WAY: Uses 'textvariable' to create a permanent bridge.
        # We explicitly label this as the "Reactive/Live" binding.
        tk.Label(self, text="CORRECT WAY: Using 'textvariable=' parameter", 
                 fg="green", font=("Arial", 8, "bold")).pack(pady=(10, 0))
        lbl_r = tk.Label(self, textvariable=self.text_val, fg="green", 
                         font=("Arial", 10, "bold"))
        lbl_r.pack(pady=2)

        # This comment explains the outcome to the student
        msg = "Note: Red label displays the Tcl memory name (PY_VAR0)."
        tk.Label(self, text=msg, font=("Arial", 7)).pack(side="bottom", pady=10)

if __name__ == "__main__":
    app = VariableBindingApp()
    app.mainloop()


**Minimal Implementation Blueprint: Live Tracking**


Below is a compact, highly educational script demonstrating a StringVar explicitly bound to an Entry widget, utilizing .trace_add() to mirror typed user input instantly into a static Label display

.
This blueprint demonstrates variable tracing in action by creating a live text-mirroring utility. When you launch the script, a small window appears containing an empty text box and a gray label that reads "Waiting for input...".
The instant you click inside the box and type a character, the background trace_add() watch guard detects the modification and fires the run_on_write() function. This function instantly extracts your text and copies it into the label, shifting its color to blue. If you backspace and erase all the text, the program automatically resets itself back to the default gray waiting message.


In [2]:
import tkinter as tk

class LiveTrackingApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Live Form Automation Blueprint")
        self.geometry("400x180")

        # Step 1: Initialize the specialized tracking variable
        # Variable "stream_var" automatically tracks changes in Entry widget 
        # It is the "Source of Truth" for our live tracking system.
        self.stream_var = tk.StringVar(value="")

        # Step 2: Establish the automated background variable trace
        # This forces an immediate jump to self.run_validation on modification
        # trace_add()→ live connect between variable changes & validation logic
        # run_validation()→ with *args having Tcl params→ prevent TypeErrors
        self.stream_var.trace_add("write", self.run_validation)

        # Step 3: Create input fields bound securely to the tracking variable
        # ent_input→ Entry widget, linked to stream_var for live tracking
        self.ent_input = tk.Entry(self, textvariable=self.stream_var, width=35)
        self.ent_input.pack(pady=20)

        # Step 4: Configure the dynamic feedback display widget
        # lbl_mirror→ displays live feedback, updated by run_validation
        self.lbl_mirror = tk.Label(self, 
                                   text="Waiting for input...", 
                                   fg="gray", font=("Arial", 11))
        self.lbl_mirror.pack()

    def run_validation(self, *args):
        # Monitors inputs & updates visual params based on current text states
        # Unpacking *args extracts Tcl parameters safely, avoiding TypeError
        current_text = self.stream_var.get()

        if current_text.strip() == "":
            self.lbl_mirror.config(text="Waiting for input...", fg="gray")
        else:
            self.lbl_mirror.config(text=f"Live Preview: {current_text}", 
                                   fg="blue")

if __name__ == "__main__":
    app = LiveTrackingApp()
    app.mainloop()


The following script shows the 3 ways to Handle Events in Tkinter

In [7]:
import tkinter as tk
from tkinter import ttk

class EventArchitectureApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Event Architecture Demo")
        self.geometry("460x280")
        
        # Static header banner. 
        ttk.Label(self, text="Interact below to test systems.",
                  font=("Arial", 10, "bold")).pack(pady=6)
        
        # Three labels, one for each system.
        # 1. lbl_cmd shows status of command system
        self.lbl_cmd = ttk.Label(self, text="Cmd Status: Idle", foreground="blue")
        self.lbl_cmd.pack(pady=1)
        
        # 2. lbl_hw shows status of hardware input
        self.lbl_hw = ttk.Label(self, text="Hardware: None", foreground="green")
        self.lbl_hw.pack(pady=1)
        
        # 3. lbl_virt shows status of virtual events
        self.lbl_virt = ttk.Label(self, text="Virtual: Idle", foreground="purple")
        self.lbl_virt.pack(pady=1)
        
        # SYSTEM 1: Command parameter link (No parentheses on method)
        # When button clicked, it calls run_command() which updates lbl_command
        ttk.Button(self, text="Standard Command Button", 
                   command=self.run_command).pack(pady=6)
        
        # SYSTEM 2: Physical hardware direct bindings
        # Links mouse movement to hardware label, passing event with .x and .y
        self.bind("<Motion>", self.track_mouse)
        self.bind("<KeyPress>", self.track_keyboard)
        
        # SYSTEM 3: Virtual logical event setup & tracking
        # Maps virtual event <<Apply>> to physical triggers→ Ctrl+S & Enter
        self.event_add("<<Apply>>", "<Control-s>", "<Control-S>", "<Return>")
        self.bind("<<Apply>>", self.run_virtual)
        
        # Practical visual hint for testing the Virtual Event
        ttk.Label(self, text="Press 'Enter' or 'Ctrl+S' to test Virtual Event", 
                  foreground="gray").pack(pady=6)
        self.focus_force()
    
    # --- Event Handler Methods ---
    # 1. Command handler: Updates its dedicated command label
    def run_command(self):
        self.lbl_cmd.config(text="Cmd Status: Activated via command=")
    
    # 2. Hardware Mouse handler: Updates the shared hardware label
    def track_mouse(self, event):
        self.lbl_hw.config(text=f"Hardware: Mouse X={event.x}, Y={event.y}")

    # 3. Hardware Keyboard handler: Updates the shared hardware label
    def track_keyboard(self, event):
        self.lbl_hw.config(text=f"Hardware: Key {event.keysym}")

    # 4. Virtual Event handler: Updates its dedicated virtual event label
    def run_virtual(self, event):
        self.lbl_virt.config(text="Virtual: Saved via <<Apply>>")

if __name__ == "__main__":
    EventArchitectureApp().mainloop()



In [8]:
import tkinter as tk
from tkinter import ttk

class MainApplication(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Container Widgets Demo")
        self.geometry("650x400")

        # 1. FRAME: Top organizational header area
        top_frame = ttk.Frame(self)
        top_frame.pack(fill="x")
        
        ttk.Label(top_frame, text="Top Frame: Tkinter Container Widgets",
                  font=("Arial", 14, "bold")).pack(pady=10)

        # 2. LABELFRAME: Titled input panel group
        user_frame = ttk.LabelFrame(self, text="Label Frame: User Details", 
                                    padding=10)
        user_frame.pack(fill="x", padx=10, pady=5)
        
        ttk.Label(user_frame, text="Name:").grid(row=0, column=0, 
                                                 padx=5, pady=5)
        ttk.Entry(user_frame).grid(row=0, column=1, padx=5, pady=5)

        # 3. NOTEBOOK: Tabbed interface engine
        my_notebook = ttk.Notebook(self)
        my_notebook.pack(fill="both", expand=True, padx=10, pady=5)
        
        tab1 = ttk.Frame(my_notebook)
        tab2 = ttk.Frame(my_notebook)
        
        my_notebook.add(tab1, text="Tab1: Home")
        my_notebook.add(tab2, text="Tab2: Settings")

        ttk.Label(tab1, text="Welcome to Tab1: Home").pack(pady=20)
        ttk.Label(tab2, text="Welcome to Tab2: Settings").pack(pady=20)

        # 4. PANEDWINDOW: Resizable horizontal split view inside Tab 1
        paned = ttk.PanedWindow(tab1, orient="horizontal")
        paned.pack(fill="both", expand=True, padx=10, pady=10)

        # tk.Frames are used specifically to support custom background colors
        left_paned = tk.Frame(paned, width=150, bg="#e8f5e9", 
                              relief="groove", bd=2)
        right_paned = tk.Frame(paned, width=150, bg="#e3f2fd", 
                               relief="groove", bd=2)

        paned.add(left_paned, weight=1)
        paned.add(right_paned, weight=1)

        ttk.Label(left_paned, text="Left Pane of resizable window").pack(
            pady=20, expand=True)
        ttk.Label(right_paned, text="Right Pane of resizable window").pack(
            pady=20, expand=True)

        # 5. TOPLEVEL: Popup function and trigger button on root window
        def open_window():
            top_window = tk.Toplevel(self)
            top_window.title("Extra Window")
            top_window.geometry("250x100")
            ttk.Label(top_window, text="This is a Toplevel Window").pack(
                pady=20)

        ttk.Button(self, text="Open Toplevel Window", 
                   command=open_window).pack(pady=10)

if __name__ == "__main__":
    app = MainApplication()
    app.mainloop()


Following script shows usage of following display and Interaction widgets: Label, Message, Canvas, Listbox and Text.
This script demonstrates several important display and interaction widgets available in Python’s tkinter library. The application combines graphical drawing, text display, list selection, and text editing within a single GUI window. It introduces the Canvas widget by allowing the user to click and change the color of a rectangle dynamically. The script also demonstrates the Message widget for displaying wrapped text and the Listbox widget for handling item selection events. In addition, the Text widget is used to create a multi-line editable text area that can be cleared through a button linked with the command= parameter. The example illustrates important GUI concepts such as event-driven programming, event binding using bind(), widget interaction, dynamic content updating, and object-oriented GUI design using classes.


In [9]:
import tkinter as tk
from tkinter import ttk

class MainApplication(tk.Tk):
    def __init__(self):
        super().__init__()  # Initialize parent class tk.Tk to set main window
        self.title("Display & Interaction Demo")
        self.geometry("600x650")
        
        # State flag for Canvas toggle. 
        self.is_blue = False # Initially gray. Click toggle to blue and back.

        # 1. Canvas (Interactive: Toggle color)
        self.canvas = tk.Canvas(self, width=200, height=60, bg="white")
        self.canvas.pack(pady=5)
        self.rect = self.canvas.create_rectangle(20, 10, 180, 50, fill="gray")
        # Bind left mouse click to on_canvas_click method. 
        # Event object with .x and .y will be passed.
        self.canvas.bind("<Button-1>", self.on_canvas_click)

        # 2. Message (Display: Long wrapped text)
        tk.Message(self, text="Message widget automatically wraps "
                   "long text inside a fixed width area.", 
                   width=300, bg="lightyellow").pack(pady=5)

        # 3. Listbox (Interactive: Selection updates label)
        self.listbox = tk.Listbox(self, height=4)
        for item in ["Python", "Java", "C++"]:
            self.listbox.insert(tk.END, item) # Insert items into the listbox
        self.listbox.pack(pady=5)
        # Bind the virtual event <<ListboxSelect>> to on_list_select method.
        # This event fires when the selection changes. Event object is passed.
        self.listbox.bind("<<ListboxSelect>>", self.on_list_select)

        self.result_lbl = ttk.Label(self, text="Selection: None")
        self.result_lbl.pack(pady=5)

        # 4. Text (Interactive: Clear button)
        self.text_box = tk.Text(self, height=4, width=30)
        self.text_box.insert("1.0", "Edit this text...")
        self.text_box.pack(pady=5)
        # Button to clear text box. Uses command= to link to clear_text method
        ttk.Button(self, text="Clear Text", 
                   command=self.clear_text).pack(pady=5)

    def on_canvas_click(self, event):
        # Toggle state and change color
        self.is_blue = not self.is_blue
        # Depending on the state, set color to skyblue or gray
        color = "skyblue" if self.is_blue else "gray"
        self.canvas.itemconfig(self.rect, fill=color)

    def on_list_select(self, event):
        idx = self.listbox.curselection()
        # If an item is selected, get its value and update the result label
        if idx:
            val = self.listbox.get(idx[0])
            self.result_lbl.config(text=f"Selection: {val}")

    def clear_text(self):
        self.text_box.delete("1.0", tk.END)

if __name__ == "__main__":
    app = MainApplication()
    app.mainloop()



The script given below demonstrates the use of important input widgets available in Python’s tkinter library. The application collects user information such as name, address, age, and department through different GUI controls. It introduces widgets like Entry for single-line input, Text for multi-line input, Spinbox for numeric selection, and Combobox for choosing items from a predefined list. The script also demonstrates event-driven programming by connecting a button to the show_data() method using the command= parameter. In addition, the example illustrates data retrieval from widgets, simple input validation, resetting widget values, focus management using focus_set(), and the use of read-only widget states to control user input. Overall, the program provides students with a practical introduction to designing interactive data-entry forms in Tkinter.


In [10]:
import tkinter as tk
from tkinter import ttk 

class MainApplication(tk.Tk):
    def __init__(self):
        super().__init__()  
        self.title("Input Widgets Demo")
        self.geometry("300x450")

        # 1. Entry widget
        tk.Label(self, text="Name (Entry Widget):").pack()
        self.ent_name = tk.Entry(self, width=30)
        self.ent_name.pack(pady=5)
        self.ent_name.focus_set()  # Automatically sets focus on start

        # 2. Text widget
        tk.Label(self, text="Address (Text Widget):").pack()
        self.txt_address = tk.Text(self, height=4, width=30)
        self.txt_address.pack(pady=5)

        # 3. Spinbox (Set to readonly to prevent negative/invalid input)
        tk.Label(self, text="Age (Spinbox Widget- readonly, 1-100):").pack()
        self.spin_age = tk.Spinbox(self, from_=1, to=100, width=10, 
                                   state="readonly")
        self.spin_age.pack(pady=5)

        # 4. Combobox
        tk.Label(self, text="Department (Combobox Widget):").pack()
        departments = ["HR", "Finance", "IT", "Sales"]
        self.combo_dept = ttk.Combobox(self, values=departments, 
                                       state="readonly")
        self.combo_dept.current(0)
        self.combo_dept.pack(pady=5)

        # 5. Button uses 'command=' to link to show_data method
        tk.Button(self, text="Show Details", 
                  command=self.show_data).pack(pady=10)  

    def show_data(self):
        # Retrieve values
        name = self.ent_name.get()
        address = self.txt_address.get("1.0", "end").strip()
        age = self.spin_age.get()
        dept = self.combo_dept.get()

        # Simple validation: ensure name is not empty
        if not name:
            print("Error: Name is required!")
            return

        # Display collected information on the console
        print(f"--- User Details ---")
        print(f"Name: {name}")
        print(f"Address: {address}")
        print(f"Age: {age}")
        print(f"Department: {dept}")
        
        # Optional: Clear Entry and Text widgets after submission
        self.ent_name.delete(0, tk.END)
        self.txt_address.delete("1.0", tk.END)
        
        # 2. Reset Spinbox. Use "Unlock-Edit-Lock" Pattern
        self.spin_age.config(state="normal")      # Temporarily unlock
        self.spin_age.delete(0, tk.END)
        self.spin_age.insert(0, "1")
        self.spin_age.config(state="readonly")    # Re-lock

        # 3. Reset Combobox. Use "Unlock-Edit-Lock" Pattern
        self.combo_dept.config(state="normal")    # Temporarily unlock
        self.combo_dept.current(0)
        self.combo_dept.config(state="readonly")  # Re-lock

if __name__ == "__main__":
    app = MainApplication()
    app.mainloop()


Error: Name is required!
Error: Name is required!
Error: Name is required!
Error: Name is required!
--- User Details ---
Name: aaa
Address: -9
Age: 6
Department: HR


The following script shows use of (1) Listbox (2) Checkbutton (3) Radiobutton (4) OptionMenu
This script demonstrates the use of various selection-oriented widgets available in Python’s tkinter library. The application allows users to make choices through controls such as Listbox, Checkbutton, Radiobutton, and OptionMenu. The Listbox widget is used for selecting one or more programming languages, while Checkbutton widgets allow multiple independent feature selections. The Radiobutton group demonstrates mutually exclusive selection, where only one size option can be chosen at a time. The OptionMenu widget provides a simple drop-down selection mechanism. The script also illustrates the use of special Tkinter variable classes such as BooleanVar() and StringVar() for tracking widget states. In addition, students are introduced to concepts like event-driven programming, retrieving selected values from widgets, multiple-selection handling, and organizing user input in GUI applications.


In [12]:
import tkinter as tk
from tkinter import ttk

class MainApplication(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Selection Widgets Demo")
        self.geometry("350x500")

        # 1. Listbox allows single or multiple selection
        tk.Label(self,text="Languages (Listbox):").pack(anchor="w",padx=10)
        # tk.MULTIPLE → multiple selection, height=4 → shows 4 items at a time
        # exportselection=False→ selections highlighted even if focus changes
        self.lb = tk.Listbox(self, selectmode=tk.MULTIPLE, height=4, 
                             exportselection=False)
        for item in ["Python", "Java", "C++", "Go"]:
            self.lb.insert(tk.END, item)
        self.lb.pack(fill="x", padx=10)

        # 2. Checkbuttons
        tk.Label(self,text="Features (Checkbutton):").pack(anchor="w",padx=10)
        self.check_vars = {}
        features = ["Dark Mode", "Notifications", "Auto-save"]
        for feature in features:
            var = tk.BooleanVar()  # BooleanVar()→ tracking of checkbox state
            self.check_vars[feature] = var
            ttk.Checkbutton(self,text=feature,
                            variable=var).pack(anchor="w",padx=10)

        # 3. Radiobuttons
        tk.Label(self,text="Size (Radiobuttons):").pack(anchor="w",padx=10)
        self.size_var = tk.StringVar(value="M") # value="M" → default selection
        for size in ["S", "M", "L", "XL"]:
            # variable=self.size_var is StringVar()→ tracks selected option
            ttk.Radiobutton(self, text=size, variable=self.size_var, 
                            value=size).pack(anchor="w", padx=10)

        # 4. OptionMenu (using tk.OptionMenu for compatibility)
        tk.Label(self, text="Options (OptionMenu):").pack(anchor="w", padx=10)
        self.option_var = tk.StringVar(value="Option 1")
        tk.OptionMenu(self, self.option_var, "Option 1", "Option 2", 
                      "Option 3").pack(fill="x", padx=10, pady=5)

        # 5. Button. When clicked, it calls self.show_selection() 
        ttk.Button(self, text="Show Selection", 
                   command=self.show_selection).pack(pady=10)

    def show_selection(self):
        langs = [self.lb.get(i) for i in self.lb.curselection()]
        features = [f for f, var in self.check_vars.items() if var.get()]
        
        print(f"\n--- Selection Summary ---")
        print(f"Languages: {langs}")
        print(f"Features: {features}")
        print(f"Size: {self.size_var.get()}")
        print(f"Option: {self.option_var.get()}")

if __name__ == "__main__":
    app = MainApplication()
    app.mainloop()



--- Selection Summary ---
Languages: ['Python', 'Java']
Features: ['Dark Mode']
Size: XL
Option: Option 1


The script given below demonstrates the use of important control widgets available in Python’s tkinter library. The application introduces widgets such as Button, Scale, and Progressbar, which are commonly used in interactive desktop applications. The Button widget is connected to a function using the command= parameter, illustrating event-driven programming and callback functions. The Scale widget allows the user to select a numerical value interactively, while the associated IntVar() object demonstrates how Tkinter variables can automatically track and display widget states. The program also showcases a Progressbar, which visually represents task progress and simulates a loading operation. In addition, the example introduces concepts such as GUI updating with update(), timed delays using after(), variable binding, and user interaction through graphical controls. Overall, the script helps students understand how control widgets are used to create responsive and dynamic GUI applications.

In [13]:
import tkinter as tk
from tkinter import ttk

class MainApplication(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Control Widgets Demo")
        self.geometry("420x360")

        # Variables: IntVar tracks the numerical state of the Scale
        self.scale_var = tk.IntVar(value=50)

        # 1. BUTTON
        tk.Label(self, text="1. BUTTON:").pack(anchor="w", padx=10)
        
        def on_click():
            print("Action triggered!")

        ttk.Button(self, text="Run Action", command=on_click).pack(padx=10)

        # 2. SCALE
        tk.Label(self, text="2. SCALE:").pack(anchor="w", padx=10)
        ttk.Scale(self, from_=0, to=100, variable=self.scale_var, 
                  orient=tk.HORIZONTAL).pack(padx=10)
        tk.Label(self, textvariable=self.scale_var).pack()

        # 3. PROGRESSBAR
        tk.Label(self, text="3. PROGRESSBAR:").pack(anchor="w", padx=10)
        self.pb = ttk.Progressbar(self, length=300, mode="determinate")
        self.pb.pack(padx=10, pady=3)

        def start_progress():
            self.pb.config(value=0)
            # We use a loop to simulate a task progressing to 100%.
            # NOTE: For-loops with update() block the GUI. In real apps, 
            # use callbacks or background tasks to keep the UI responsive.
            for i in range(0, 101, 10):
                self.pb.config(value=i)
                self.update()      # Force immediate GUI redraw
                self.after(200)    # Pause for 200ms to visualize progress
            print("Done!")

        ttk.Button(self, text="Start Loading", 
                   command=start_progress).pack(padx=10)

if __name__ == "__main__":
    app = MainApplication()
    app.mainloop()



Done!
Action triggered!


In [14]:
import tkinter as tk
from tkinter import messagebox

class MessageboxDemo(tk.Tk):

    def __init__(self):
        super().__init__()
        self.title("Messagebox Demo")

        # Create 2 buttons: (1) "Show Message" (2) "Exit"
        # The methods are called via the self reference prefix
        tk.Button(self, text="Show Message", command=self.show_info).pack()
        tk.Button(self, text="Exit", command=self.ask_exit).pack()

    def show_info(self):  # Show information message box
        answer_showinfo = messagebox.showinfo("Information", 
                                              "Welcome to Tkinter")
        print("showinfo answer:", answer_showinfo)  # showinfo returns "ok"

    def ask_exit(self):  # Ask user before closing the application
        answer_exit = messagebox.askyesno("Exit", "Do you want to quit?")
        print("ask_exit answer:", answer_exit)  # askyesno returns True/False
        if answer_exit:
            self.destroy()

if __name__ == "__main__":
    # Initialize and execute the object-oriented window lifecycle loop
    app = MessageboxDemo()
    app.mainloop()


showinfo answer: ok
ask_exit answer: True


The filedialog module allows users to open or save files using a standard file browser window. Instead of typing file paths manually, users can select files visually.
You can import filedialog module from tkinter as: from tkinter import filedialog
The two common filedialog functions are: (1) askopenfilename()→ Select file to open and (2) asksaveasfilename()→ Select file to save
The following script demonstrates the use of file dialog boxes in Python’s tkinter library. File dialogs provide a standard graphical interface that allows users to select files for opening or specify filenames for saving data. The application contains two buttons: one for opening an existing file and another for choosing a location to save a file.
The script introduces the functions askopenfilename() and asksaveasfilename() from the filedialog module. It also demonstrates the use of important parameters such as title, filetypes, defaultextension, and parent to create user-friendly and well-controlled dialog windows. The selected filenames are retrieved and displayed through the console output.
In addition, the example illustrates event-driven programming using the command= parameter, object-oriented GUI design using classes, and interaction between the application window and operating-system file management tools. Overall, this program helps students understand how GUI applications can interact with the file system in a professional and user-friendly manner.


In [15]:
import tkinter as tk
from tkinter import filedialog

class FileDialogDemo(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("File Dialog Demo")
        self.geometry("300x150")

        # Buttons with padding for better visual layout
        tk.Button(self, text="Open File", 
                  command=self.open_file).pack(pady=10)
        
        tk.Button(self, text="Save File", 
                  command=self.save_file).pack(pady=5)

    def open_file(self):
        # Using parent=self ensures the dialog stays on top of our app
        # Filtering files makes the interface much more user-friendly
        filename = filedialog.askopenfilename(
            parent=self,
            title="Select a Text File",
            filetypes=[("Text files", "*.txt"), ("All files", "*.*")]
        )
        if filename:
            print("Selected File:", filename)

    def save_file(self):
        # Save dialogs also benefit from 'parent' and 'defaultextension'
        filename = filedialog.asksaveasfilename(
            parent=self,
            title="Save your work",
            defaultextension=".txt",
            filetypes=[("Text files", "*.txt")]
        )
        if filename:
            print("Save File to:", filename)

if __name__ == "__main__":
    app = FileDialogDemo()
    app.mainloop()


This program demonstrates the creation and management of multiple windows in Python’s tkinter library using the Toplevel widget. The application consists of a main window that can open a separate child window dynamically through a button click. The example helps students understand how GUI applications can manage independent windows within the same program.

The script also introduces object-oriented GUI programming by defining separate classes for the main application window and the child window. The ChildWindow class inherits from tk.Toplevel, showing how additional windows can be created as independent GUI objects linked to the main application.

An important concept demonstrated in this example is object destruction and memory management. The script defines the special __del__() destructor method, which is automatically called by Python when the child window object is removed from memory. By observing the terminal output after closing the child window, students can better understand object lifecycle, garbage collection, and resource cleanup in Python applications.

Overall, this example introduces concepts such as multiple-window GUI design, inheritance, event-driven programming, object creation and destruction, and the relationship between graphical objects and memory management.

In [16]:
import tkinter as tk

class ChildWindow(tk.Toplevel):
    def __init__(self, master):
        super().__init__(master)
        self.title("Child Window")
        tk.Label(self, text="Watch the terminal when you close me!").pack(pady=20)
        tk.Button(self, text="Close", command=self.destroy).pack()

    # This method is called by Python the moment the object is removed from memory
    def __del__(self):  
        print("MEMORY ALERT: ChildWindow object has been deleted from memory.")

class MainApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Main Window")
        self.geometry("300x200")
        tk.Button(self, text="Open Window", command=self.open_child).pack(pady=50)

    def open_child(self):
        ChildWindow(self)

if __name__ == "__main__":
    app = MainApp()
    app.mainloop()


MEMORY ALERT: ChildWindow object has been deleted from memory.


This program demonstrates the use of error handling and validation techniques in a Python tkinter GUI application. The application accepts age input from the user and checks whether the entered value is valid before processing it further. The example shows how graphical applications can safely handle incorrect or unexpected user input without crashing.

The script introduces the use of try, except, else, and finally blocks for structured exception handling. It validates user input by checking for empty values, non-numeric input, and negative numbers, while raising and handling ValueError exceptions where necessary. The program also demonstrates the use of message boxes such as showwarning(), showerror(), and showinfo() to communicate with the user in a clear and user-friendly manner.

An additional important concept illustrated in this example is application logging using Python’s logging module. Errors are recorded in an external log file (app.log) along with timestamps and technical details, helping developers diagnose problems more effectively.

Overall, this example helps students understand GUI validation, exception handling, defensive programming, logging, and the importance of building reliable and user-friendly applications.

In [17]:
import tkinter as tk
from tkinter import messagebox
import logging

# Setup logging: Creates 'app.log' in the same folder
logging.basicConfig(filename='app.log', level=logging.ERROR, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

class ErrorHandlingApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Tkinter Error Handling")
        self.geometry("300x180")

        tk.Label(self, text="Enter your age").pack(pady=5)
        self.entry = tk.Entry(self)
        self.entry.pack(pady=5)

        tk.Button(self, text="Submit", command=self.check_age).pack(pady=10)

    def check_age(self):
        age_text = self.entry.get().strip()

        if age_text == "":
            messagebox.showwarning("Warning", "Please enter age")
            return

        try:
            age = int(age_text)
            if age < 0:
                raise ValueError("Age cannot be negative")

        except ValueError as e:
            # 1. Show user-friendly error
            messagebox.showerror("Error", f"Invalid input:\n{e}")
            
            # 2. Log technical details for the developer
            logging.error(f"Validation Error: {e}", exc_info=True)

        else:
            messagebox.showinfo("Success", f"Age entered: {age}")

        finally:
            print("Input checked")

if __name__ == "__main__":
    app = ErrorHandlingApp()
    app.mainloop()


Input checked
